# Notebook 5: Innovation 2 — LoRA Adapters for LAPT

## What is this innovation?

**Problem with LAPT**: It fine-tunes ALL 110M parameters of mBERT.
This is:
- Computationally expensive
- Prone to catastrophic forgetting (model forgets other languages)
- Data-hungry (needs a lot of unlabeled text to be effective)

**Our solution**: Use **LoRA (Low-Rank Adaptation)** from the PEFT library.

LoRA adds small trainable matrices to each attention layer:
```
W_adapted = W_original + alpha * (A @ B)
```
where A ∈ R^(d×r) and B ∈ R^(r×d) with rank r << d.

- **Original mBERT LAPT**: trains ~110M parameters
- **LoRA-LAPT**: trains only ~0.3-1M parameters (0.3-1% of the model)

## Why is this a valid thesis contribution?

LoRA (Hu et al., 2021) was published AFTER the original paper (2020).
The paper could not have used it. We ask:

1. Can LoRA-LAPT match full-LAPT with far fewer trainable parameters?
2. Does LoRA reduce catastrophic forgetting of other languages?
3. Is LoRA-LAPT more data-efficient (works with less unlabeled text)?

This is a practical contribution: practitioners in low-resource settings
often can't afford to fine-tune 110M parameters.

## Reference
Hu, E. J., et al. (2022). LoRA: Low-Rank Adaptation of Large Language Models. ICLR 2022.

In [ ]:
# Environment setup
import os, sys, json
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR  = '/content/parsing-mbert'
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/Sansgithub22/mtechthesis2Parsing-mbert.git {REPO_DIR}')

sys.path.insert(0, os.path.join(REPO_DIR, 'thesis', 'src'))

# Colab already has PyTorch — only install the missing packages
!pip install -q transformers==4.38.0 tokenizers==0.15.2 datasets==2.18.0 peft==0.10.0 accelerate==0.27.0 conllu==4.5.3 sentencepiece==0.1.99 tqdm seaborn

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# LoRA-LAPT implementation
# We extend the LAPT idea but only train LoRA adapter weights

import os, math
import torch
from transformers import (
    AutoModelForMaskedLM, AutoTokenizer,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset


def run_lora_lapt(
    model_name: str,
    train_text_path: str,
    eval_text_path: str,
    output_dir: str,
    lora_rank: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.1,
    num_epochs: int = 5,
    batch_size: int = 16,
    learning_rate: float = 2e-4,   # LoRA uses higher LR than full fine-tuning
    warmup_steps: int = 500,
    max_length: int = 128,
    fp16: bool = True,
):
    """
    Run Language-Adaptive Pre-Training using LoRA.

    Only the LoRA matrices are trained (~0.3% of parameters).
    This is much faster and uses less memory than full LAPT.

    Args:
        lora_rank: rank of the low-rank matrices (r). Higher = more capacity.
        lora_alpha: scaling factor (typically 2*rank)
        lora_dropout: dropout in LoRA layers
        learning_rate: higher than full fine-tuning (2e-4 recommended for LoRA)
    """
    print(f'\n{"="*60}')
    print(f'LoRA-LAPT | Model: {model_name}')
    print(f'LoRA rank={lora_rank}, alpha={lora_alpha}, epochs={num_epochs}')
    print(f'{"="*60}\n')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForMaskedLM.from_pretrained(model_name)

    # Apply LoRA configuration
    # We adapt the query and value projection matrices in all attention layers
    lora_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=['query', 'value'],  # adapt Q and V attention matrices
        bias='none',
        inference_mode=False,
    )

    model = get_peft_model(base_model, lora_config)

    # Print parameter efficiency
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total params:     {total:>12,}')
    print(f'Trainable params: {trainable:>12,} ({100*trainable/total:.2f}%)')

    # Load data
    def load_lines(path):
        with open(path) as f:
            return [l.strip() for l in f if l.strip()]

    train_dataset = Dataset.from_dict({'text': load_lines(train_text_path)})
    eval_dataset  = Dataset.from_dict({'text': load_lines(eval_text_path)})

    def tokenize(examples):
        return tokenizer(examples['text'], truncation=True,
                         max_length=max_length, padding=False)

    train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
    eval_dataset  = eval_dataset.map(tokenize, batched=True, remove_columns=['text'])

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=0.15
    )

    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        fp16=fp16 and torch.cuda.is_available(),
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=100,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    trainer.train()

    # Save: merge LoRA weights into the base model for easy loading
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    print(f'\nLoRA-LAPT complete. Merged model saved to: {output_dir}')
    return output_dir

print('LoRA-LAPT function defined.')

In [ ]:
# Data paths
UD_BASE = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')
TREEBANKS = {
    'ga':   {'train': f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-test.conllu'},
    'mt':   {'train': f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-test.conllu'},
    'vi':   {'train': f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-test.conllu'},
    'sing': {'train': f'{REPO_DIR}/data/sing/train.conllu',
             'dev':   f'{REPO_DIR}/data/sing/dev.conllu',
             'test':  f'{REPO_DIR}/data/sing/test.conllu'},
}
UNLABELED = {
    'ga':   f'{REPO_DIR}/data/unlabeled/ga',
    'mt':   f'{REPO_DIR}/data/unlabeled/mt',
    'vi':   f'{REPO_DIR}/data/unlabeled/vi',
    'sing': f'{REPO_DIR}/data/unlabeled/sg',
}
print('Paths configured.')

In [ ]:
# Shared parser experiment runner
from transformers import AutoTokenizer
from data.conllu_reader import read_conllu, get_relation_vocab
from models.encoder import BERTEncoder
from models.biaffine_parser import BiaffineParser
from training.trainer import ParserTrainer

def run_parser_experiment(lang, model_path, freeze_bert, experiment_name,
                          max_epochs=200, patience=20, batch_size=16):
    paths = TREEBANKS[lang]
    train_sents = read_conllu(paths['train'])
    dev_sents   = read_conllu(paths['dev'])
    test_sents  = read_conllu(paths['test'])
    rel_vocab   = get_relation_vocab(train_sents)
    n_rels      = max(rel_vocab.values()) + 1
    tokenizer   = AutoTokenizer.from_pretrained(model_path)
    encoder     = BERTEncoder(model_name=model_path, freeze=freeze_bert, dropout=0.1)
    model       = BiaffineParser(encoder, n_rels, arc_dim=500, rel_dim=100,
                                 bilstm_layers=3, bilstm_hidden=400,
                                 mlp_dropout=0.33, use_bilstm=freeze_bert)
    save_dir = os.path.join(WORKSPACE, 'results', experiment_name, lang)
    os.makedirs(save_dir, exist_ok=True)
    trainer = ParserTrainer(model, train_sents, dev_sents, tokenizer, rel_vocab,
                            save_dir, batch_size=batch_size, max_epochs=max_epochs,
                            patience=patience, device=DEVICE)
    best_dev_las = trainer.train()
    test_las, test_uas = trainer.evaluate_test(test_sents, tokenizer)
    result = {'lang': lang, 'experiment': experiment_name,
              'best_dev_las': round(best_dev_las, 2),
              'test_las': round(test_las, 2), 'test_uas': round(test_uas, 2)}
    with open(os.path.join(save_dir, 'result.json'), 'w') as f:
        json.dump(result, f, indent=2)
    return result

print('Parser runner loaded.')

In [ ]:
# ============================================================
# Run LoRA-LAPT experiment
# ============================================================
LANG = 'mt'   # Change to run different languages
LAPT_EPOCHS = {'ga': 5, 'mt': 20, 'vi': 5, 'sing': 5}

lora_model_dir = os.path.join(WORKSPACE, 'lora_models', f'lora_lapt_{LANG}')

if os.path.exists(os.path.join(lora_model_dir, 'config.json')):
    print(f'LoRA-LAPT model for {LANG} already exists.')
else:
    lora_model_dir = run_lora_lapt(
        model_name='bert-base-multilingual-cased',
        train_text_path=os.path.join(UNLABELED[LANG], 'train.txt'),
        eval_text_path=os.path.join(UNLABELED[LANG], 'valid.txt'),
        output_dir=lora_model_dir,
        lora_rank=16,
        lora_alpha=32,
        num_epochs=LAPT_EPOCHS[LANG],
        batch_size=16,
        learning_rate=2e-4,
        fp16=True,
    )

print(f'LoRA-LAPT model: {lora_model_dir}')

In [ ]:
# Train parser on LoRA-LAPT model
lora_result = run_parser_experiment(
    lang=LANG,
    model_path=lora_model_dir,
    freeze_bert=False,
    experiment_name='lora_lapt_ft',
)
print(f'LoRA-LAPT+FT [{LANG}]: LAS = {lora_result["test_las"]}')

# Compare with full LAPT
PAPER_LAPT = {'ga': 75.45, 'mt': 82.77, 'vi': 67.50, 'sing': 79.30}
print(f'Paper LAPT+FT [{LANG}]: LAS = {PAPER_LAPT.get(LANG)}')
print(f'Difference: {lora_result["test_las"] - PAPER_LAPT.get(LANG, 0):+.2f}')

In [ ]:
# ============================================================
# LoRA Rank Ablation Study
# ============================================================
# Test different LoRA ranks to find the sweet spot
# This is valuable analysis for your thesis!

# NOTE: Run this only after you have enough Colab GPU time
# Each rank experiment takes ~30-60 min

ABLATION_LANG = 'mt'   # Use Maltese (largest improvement in paper)
RANKS = [4, 8, 16, 32]  # ranks to test

ablation_results = {}

for rank in RANKS:
    print(f'\n--- LoRA rank = {rank} ---')

    rank_model_dir = os.path.join(WORKSPACE, 'lora_ablation', f'rank_{rank}_{ABLATION_LANG}')

    if not os.path.exists(os.path.join(rank_model_dir, 'config.json')):
        run_lora_lapt(
            model_name='bert-base-multilingual-cased',
            train_text_path=os.path.join(UNLABELED[ABLATION_LANG], 'train.txt'),
            eval_text_path=os.path.join(UNLABELED[ABLATION_LANG], 'valid.txt'),
            output_dir=rank_model_dir,
            lora_rank=rank,
            lora_alpha=rank * 2,
            num_epochs=5,  # fewer epochs for ablation
            batch_size=16,
        )

    r = run_parser_experiment(
        lang=ABLATION_LANG, model_path=rank_model_dir,
        freeze_bert=False, experiment_name=f'lora_rank{rank}_ft',
    )
    ablation_results[rank] = r['test_las']

print('\nRank Ablation Results:')
print(f'{"LoRA rank":<12} {"LAS":>8}')
for rank, las in sorted(ablation_results.items()):
    n_params = rank * 768 * 2 * 12 * 2  # approximate
    print(f'{rank:<12} {las:>8.2f}  ({n_params/1e6:.2f}M trainable params)')

In [ ]:
# ============================================================
# Final comparison: Full LAPT vs LoRA-LAPT
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

PAPER_RESULTS = {
    'MBERT+FT': {'ga': 72.67, 'mt': 76.74, 'vi': 66.13, 'sing': 78.24},
    'LAPT+FT (Paper)': {'ga': 75.45, 'mt': 82.77, 'vi': 67.50, 'sing': 79.30},
}

langs = ['ga', 'mt', 'vi', 'sing']
lang_labels = ['Irish\n(Type 1)', 'Maltese\n(Type 2)', 'Vietnamese\n(Type 0)', 'Singlish\n(Type 0)']

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(langs))
width = 0.25

bars1 = ax.bar(x - width, [PAPER_RESULTS['MBERT+FT'][l] for l in langs],
               width, label='mBERT+FT (baseline)', color='#4472C4')
bars2 = ax.bar(x, [PAPER_RESULTS['LAPT+FT (Paper)'][l] for l in langs],
               width, label='Full LAPT+FT (paper)', color='#ED7D31')

# Add LoRA results if available
lora_las = []
for lang in langs:
    p = os.path.join(WORKSPACE, 'results', 'lora_lapt_ft', lang, 'result.json')
    if os.path.exists(p):
        with open(p) as f:
            lora_las.append(json.load(f)['test_las'])
    else:
        lora_las.append(0)

if any(v > 0 for v in lora_las):
    bars3 = ax.bar(x + width, lora_las, width,
                   label='LoRA-LAPT+FT (ours)', color='#70AD47')

ax.set_xlabel('Language', fontsize=13)
ax.set_ylabel('LAS Score', fontsize=13)
ax.set_title('Dependency Parsing: mBERT vs LAPT vs LoRA-LAPT', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(lang_labels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(55, 90)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plot_path = os.path.join(WORKSPACE, 'results', 'innovation2_lora_comparison.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to: {plot_path}')